In [20]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

##############################################
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rc("font", family="Malgun Gothic")
mpl.rc("axes", unicode_minus=False)
##############################################

# 결과 저장용 임시 폴더 (이 노트북 옆에 'd009_outputs/' 가 만들어집니다)
OUT_DIR = Path("d009_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR.resolve())

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.5.0
pandas: 3.0.3
저장 폴더: C:\Users\hgwha\Desktop\workspace1\ai-data-bootcamp\D009\d009_outputs


In [21]:
# 예제: 품질 리포트 함수 v2 — 수치형 컬럼에 IQR 이상치 비율을 추가
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base



In [31]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터
np.random.seed(13)
n_days = 90
n_stores = 6
n_rows = n_days * n_stores * 4   # 매장 x 일자 x 시간대(4구간)

stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

bakery = pd.DataFrame({
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),
    "item": np.random.choice(items, n_rows),
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

print("빵셀러 데이터 생성 완료:", bakery.shape)
bakery.head()
display(bakery)
bakery.describe()

빵셀러 데이터 생성 완료: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0
...,...,...,...,...,...,...
2159,S06,2025/04/01,크루아상,5,8500,42500.0
2160,S01,20250401,쿠키,2,12000,24000.0
2161,S02,2025-04-01,케이크,2,12000,24000.0
2162,S03,2025-04-01,쿠키,1,5500,5500.0


,quantity,unit_price,revenue
count,2164.000000,2164.000000,2104.000000
mean,9.249538,6775.878004,64157.557034
std,16.868632,3080.016334,132558.433306
min,1.000000,3500.000000,-45000.000000
25%,1.000000,4500.000000,8500.000000
50%,2.000000,5500.000000,13500.000000
75%,5.000000,8500.000000,25500.000000
max,50.000000,12000.000000,600000.000000


In [22]:
# 시나리오 1 — 모범 답안
qr_bakery = quality_report_full(bakery, "bakery")
qr_bakery

[품질 리포트(완전판)] bakery
  행 수: 2,164  /  열 수: 6
  완전 중복 행: 321건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
store_id,str,0,0.00,7,NaN,False
date_str,str,0,0.00,9,NaN,False
item,str,0,0.00,6,NaN,False
quantity,int64,0,0.00,5,14.56,False
unit_price,int64,0,0.00,5,0.00,False
revenue,float64,60,2.77,26,17.30,False


In [23]:
# 시나리오 2 — 모범 답안
def b_dedup(df): return df.drop_duplicates().reset_index(drop=True)
def b_strip_store(df): return df.assign(store_id=df["store_id"].str.strip())
def b_unify_item(df): return df.assign(item=df["item"].replace({"케익": "케이크"}))
def b_parse_date(df): return df.assign(
    date=pd.to_datetime(df["date_str"], format="mixed", errors="coerce")
).drop(columns=["date_str"])

refunds_bakery = bakery[bakery["revenue"] < 0].copy()

bakery_clean = (
    bakery
    .pipe(b_dedup)
    .pipe(b_strip_store)
    .pipe(b_unify_item)
    .pipe(b_parse_date)
    .pipe(lambda d: d[d["revenue"] >= 0])    # 환불 분리 후 본 분석에서 제외
    .pipe(lambda d: d.dropna(subset=["revenue"]))
    .reset_index(drop=True)
)

print(bakery.isna().sum())

print(f"정제 전: {bakery.shape}  →  정제 후: {bakery_clean.shape}")
print(f"환불 보관: {len(refunds_bakery)}건")
print(f"item 종류: {bakery_clean['item'].unique()}")

store_id       0
date_str       0
item           0
quantity       0
unit_price     0
revenue       60
dtype: int64
정제 전: (2164, 6)  →  정제 후: (1784, 6)
환불 보관: 1건
item 종류: <ArrowStringArray>
['쿠키', '케이크', '샌드위치', '크루아상', '식빵']
Length: 5, dtype: str


In [24]:
# 시나리오 3 — 모범 답안

# KPI 1: 월별·매장별 매출 합계 (wide)
bakery_clean["month"] = bakery_clean["date"].dt.to_period("M").astype(str)
store_month = (
    bakery_clean.groupby(["store_id", "month"])["revenue"].sum().unstack(fill_value=0).round(0)
)
print("월별·매장별 매출:")
display(store_month)

# KPI 2: 상품별 평균 단가·총매출
item_kpi = (
    bakery_clean.groupby("item")
    .agg(
        avg_price=("unit_price", "mean"), 
        total_revenue=("revenue", "sum"), 
        n_orders=("revenue", "count"))
    .round(0)
    .sort_values("total_revenue", ascending=False)
)
print("\n상품별 KPI:")
display(item_kpi)

# Parquet 저장
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet")
item_kpi.to_parquet(OUT_DIR / "bakery_item_kpi.parquet")
print(f"\nParquet 저장 완료: {OUT_DIR.resolve()}")

# csv 저장
store_month.to_csv(OUT_DIR / "bakery_store_month.csv")
item_kpi.to_csv(OUT_DIR / "bakery_item_kpi.csv")
print(f"\nCsv 저장 완료: {OUT_DIR.resolve()}")

# 용량 측정
pq_size_1 = Path(OUT_DIR /"bakery_store_month.parquet").stat().st_size / 1024
pq_size_2 = Path(OUT_DIR /"bakery_item_kpi.parquet").stat().st_size / 1024
csv_size_1 = Path(OUT_DIR /"bakery_store_month.csv").stat().st_size / 1024
csv_size_2 = Path(OUT_DIR /"bakery_item_kpi.csv").stat().st_size / 1024

print(f"[파일 용량]  CSV: {csv_size_1:>7.1f} KB  /  Parquet: {pq_size_1:>7.1f} KB  (Parquet은 CSV의 {pq_size_1/csv_size_1*100:.1f}%)")
print(f"[파일 용량]  CSV: {csv_size_2:>7.1f} KB  /  Parquet: {pq_size_2:>7.1f} KB  (Parquet은 CSV의 {pq_size_2/csv_size_2*100:.1f}%)")

월별·매장별 매출:


month,2025-04,2025-05,2025-06
store_id,,,
S01,6834000.0,7243500.0,5884000.0
S02,7531500.0,6768500.0,7996500.0
S03,7424500.0,4548000.0,7297500.0
S04,6471500.0,7105000.0,6744500.0
S05,8681500.0,4102000.0,4960000.0
S06,9156000.0,5095500.0,6796500.0



상품별 KPI:


,avg_price,total_revenue,n_orders
item,,,
쿠키,6878.0,26690000.0,365
샌드위치,6555.0,26297000.0,370
케이크,6859.0,23788500.0,359
식빵,6555.0,22997500.0,355
크루아상,6946.0,20867500.0,335



Parquet 저장 완료: C:\Users\hgwha\Desktop\workspace1\ai-data-bootcamp\D009\d009_outputs

Csv 저장 완료: C:\Users\hgwha\Desktop\workspace1\ai-data-bootcamp\D009\d009_outputs
[파일 용량]  CSV:     0.2 KB  /  Parquet:     3.2 KB  (Parquet은 CSV의 1341.8%)
[파일 용량]  CSV:     0.2 KB  /  Parquet:     3.3 KB  (Parquet은 CSV의 1636.3%)


In [25]:
# 예제: 결정 로그를 리스트로 누적하다가 마지막에 표로 출력
decisions = []

def log_decision(step: str, action: str, reason: str, result: str):
    decisions.append({"step": step, "action": action, "reason": reason, "result": result})


# Part 3에서 했던 결정을 손으로 기록 (실무에서는 단계 함수 안에서 호출)
log_decision(
    "1. 중복 제거",
    "완전 중복 행 drop",
    "동일 키 동일 값은 시스템 입력 오류로 간주",
    "321건 제거",
)
log_decision(
    "2. 문자열 정제",
    "store_id strip",
    "S01 매장 id에 공백 존재",
    "S01 매장 id 공백 제거 10건",
)
log_decision(
    "3. Item 통일",
    "케익 → 케이크",
    "오입력",
    "케익 8건 케이크로 수정",
)

log_decision(
    "4. 날짜 파싱",
    "date_str를 datetime으로",
    "월별 집계가 분석 목적이므로 datetime 필요",
    "datetime 파싱 실패 0건",
)

log_decision(
    "5. 이상치 처리",
    "음수 revenue 분리 및 제거",
    "매출 분석이 목적이므로 환불은 별도 보관",
    "음수 revenue 1건 제거",
)
log_decision(
    "6. 결측치 처리",
    "revenue 결측 행 제거",
    "결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지",
    "58건 제거",
)

decision_log = pd.DataFrame(decisions)
decision_log

,step,action,reason,result
0,1. 중복 제거,완전 중복 행 drop,동일 키 동일 값은 시스템 입력 오류로 간주,321건 제거
1,2. 문자열 정제,store_id strip,S01 매장 id에 공백 존재,S01 매장 id 공백 제거 10건
2,3. Item 통일,케익 → 케이크,오입력,케익 8건 케이크로 수정
3,4. 날짜 파싱,date_str를 datetime으로,월별 집계가 분석 목적이므로 datetime 필요,datetime 파싱 실패 0건
4,5. 이상치 처리,음수 revenue 분리 및 제거,매출 분석이 목적이므로 환불은 별도 보관,음수 revenue 1건 제거
5,6. 결측치 처리,revenue 결측 행 제거,"결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지",58건 제거


In [26]:
# 예제: 결정 로그를 마크다운 표로 출력해 노트북 끝(또는 별도 파일)에 박기
md_table = "| 단계 | 액션 | 근거 | 결과 |\n|---|---|---|---|\n"
for d in decisions:
    md_table += f"| {d['step']} | {d['action']} | {d['reason']} | {d['result']} |\n"

print(md_table)

| 단계 | 액션 | 근거 | 결과 |
|---|---|---|---|
| 1. 중복 제거 | 완전 중복 행 drop | 동일 키 동일 값은 시스템 입력 오류로 간주 | 321건 제거 |
| 2. 문자열 정제 | store_id strip | S01 매장 id에 공백 존재 | S01 매장 id 공백 제거 10건 |
| 3. Item 통일 | 케익 → 케이크 | 오입력 | 케익 8건 케이크로 수정 |
| 4. 날짜 파싱 | date_str를 datetime으로 | 월별 집계가 분석 목적이므로 datetime 필요 | datetime 파싱 실패 0건 |
| 5. 이상치 처리 | 음수 revenue 분리 및 제거 | 매출 분석이 목적이므로 환불은 별도 보관 | 음수 revenue 1건 제거 |
| 6. 결측치 처리 | revenue 결측 행 제거 | 결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지 | 58건 제거 |

